# Red-Black Trees

A **Red-Black Tree** is a self-balancing binary search tree where each node has an extra bit for color (RED or BLACK). The tree uses these colors to ensure the tree remains approximately balanced during insertions and deletions.

---

## Table of Contents
1. [Red-Black Tree Properties](#1-red-black-tree-properties)
2. [Why Red-Black vs AVL?](#2-why-red-black-vs-avl)
3. [Tree Structure Visualization](#3-tree-structure-visualization)
4. [Implementation](#4-implementation)
5. [Insert with Color Fixing](#5-insert-with-color-fixing)
6. [Rotations with Color Changes](#6-rotations-with-color-changes)
7. [Examples](#7-examples)
8. [Complexity Analysis](#8-complexity-analysis)

---

[← Previous: AVL Trees](../../Tier_4_Algorithms_and_Data_Structures/05_Tree_Structures/02_avl_trees.ipynb) | [Next: Hash Tables and Bloom Filters →](../../Tier_4_Algorithms_and_Data_Structures/06_Hash_Based_Structures/01_hash_tables_bloom.ipynb)

---

---

## 1. Red-Black Tree Properties

A Red-Black Tree must satisfy **5 essential properties**:

| # | Property | Description |
|---|----------|-------------|
| 1 | **Color Property** | Every node is either **RED** or **BLACK** |
| 2 | **Root Property** | The root is always **BLACK** |
| 3 | **Leaf Property** | Every leaf (NIL/null) is **BLACK** |
| 4 | **Red Property** | If a node is **RED**, both its children must be **BLACK** (no red-red parent-child) |
| 5 | **Black-Height Property** | Every path from root to any leaf has the **same number of BLACK nodes** |

### Visual Summary:

```
Property 1: Every node is RED or BLACK
┌─────────────────────────────────────┐
│   (B)20         Each node has a     │
│   /   \         color attribute     │
│ (R)10 (R)30                         │
└─────────────────────────────────────┘

Property 2: Root is always BLACK
┌─────────────────────────────────────┐
│   (B)20  ← Root MUST be BLACK       │
│   /   \                             │
│ (R)10 (R)30                         │
└─────────────────────────────────────┘

Property 3: Leaves (NIL) are BLACK
┌─────────────────────────────────────┐
│       (B)20                         │
│       /   \                         │
│    (R)10  (R)30                     │
│    /  \    /  \                     │
│  NIL  NIL NIL NIL  ← All BLACK      │
└─────────────────────────────────────┘

Property 4: RED node → BLACK children
┌─────────────────────────────────────┐
│ VALID:          INVALID:            │
│   (R)10           (R)10             │
│   /   \           /   \             │
│ (B)5  (B)15    (R)5  (B)15  ✗       │
│                 ↑ RED-RED!          │
└─────────────────────────────────────┘

Property 5: Same BLACK count on all paths
┌─────────────────────────────────────┐
│         (B)20                       │
│         /   \                       │
│      (R)10  (B)30                   │
│      /   \                          │
│   (B)5  (B)15                       │
│                                     │
│ Path 20→10→5→NIL:   BLACK=2 (20,5)  │
│ Path 20→10→15→NIL:  BLACK=2 (20,15) │
│ Path 20→30→NIL:     BLACK=2 (20,30) │
└─────────────────────────────────────┘
```

### Why These Properties Matter

These 5 properties **guarantee** that:
- The longest path from root to leaf is **at most twice** the shortest path
- The tree height is **O(log n)**
- Search, insert, and delete all run in **O(log n)** time

---

## 2. Why Red-Black vs AVL?

Both AVL and Red-Black trees are self-balancing BSTs, but they make different trade-offs:

### Comparison Table

| Aspect | AVL Tree | Red-Black Tree |
|--------|----------|----------------|
| **Balance Factor** | Height difference ≤ 1 | Color-based (looser) |
| **Height Bound** | ≤ 1.44 log(n) | ≤ 2 log(n) |
| **Search Speed** | Faster (shorter height) | Slightly slower |
| **Insert/Delete** | More rotations needed | Fewer rotations |
| **Memory** | +1 int (balance factor) | +1 bit (color) |
| **Best For** | Read-heavy workloads | Write-heavy workloads |
| **Real-World Use** | Databases indexes | Java TreeMap, C++ std::map |

### Visual Comparison

```
Same 7 nodes in AVL vs Red-Black:

AVL Tree (strictly balanced):      Red-Black Tree (loosely balanced):

        4                                    4(B)
       / \                                  /    \
      2   6                              2(R)    6(R)
     / \ / \                            /  \    /  \
    1  3 5  7                        1(B) 3(B) 5(B) 7(B)

Height: 3                            Height: 3 (can be up to 4)
Max rotations per insert: 2          Max rotations per insert: 2
But AVL rotates more often!          Recoloring often sufficient
```

### When to Choose Which?

```
Choose AVL when:
├── Lookups >> Insertions/Deletions
├── Need fastest possible search
└── Example: Dictionary lookup, database indexes

Choose Red-Black when:
├── Many insertions and deletions
├── Need good overall performance
└── Example: Language runtime libraries (Java, C++, Linux kernel)
```

---

## 3. Tree Structure Visualization

### Red-Black Tree Structure

Using **(R)** for RED nodes and **(B)** for BLACK nodes:

```
                    (B)13
                   /     \
               (R)8      (R)17
               /  \      /   \
            (B)1  (B)11 (B)15 (B)25
              \             \    \
             (R)6         (R)22 (R)27
```

### Verifying Properties:

```
✓ Property 1: All nodes are R or B
✓ Property 2: Root (13) is BLACK
✓ Property 3: All NIL leaves are BLACK (implicit)
✓ Property 4: RED nodes (8,17,6,22,27) have BLACK children
✓ Property 5: Black-height = 2 on all paths

Black-height verification:
  13→8→1→NIL      = 2 black (13, 1)
  13→8→1→6→NIL    = 2 black (13, 1)
  13→8→11→NIL     = 2 black (13, 11)
  13→17→15→NIL    = 2 black (13, 15)
  13→17→25→22→NIL = 2 black (13, 25)
  13→17→25→27→NIL = 2 black (13, 25)
```

---

## 4. Implementation

Let's implement a Red-Black Tree with full insertion support.

In [ ]:
# Color constants
RED = True
BLACK = False


class RBNode:
    """Node for Red-Black Tree with color attribute."""
    
    def __init__(self, data, color=RED):
        self.data = data
        self.color = color  # New nodes are RED by default
        self.parent = None
        self.left = None
        self.right = None
    
    def __repr__(self):
        color_str = "R" if self.color == RED else "B"
        return f"({color_str}){self.data}"
    
    def is_red(self):
        return self.color == RED
    
    def is_black(self):
        return self.color == BLACK

In [ ]:
class RedBlackTree:
    """Red-Black Tree implementation with insert and fix-up."""
    
    def __init__(self):
        # NIL sentinel node (always BLACK)
        self.NIL = RBNode(data=None, color=BLACK)
        self.root = self.NIL
    
    # ==================== Helper Methods ====================
    
    def is_nil(self, node):
        """Check if node is NIL sentinel."""
        return node == self.NIL or node is None
    
    def is_root(self, node):
        """Check if node is the root."""
        return node.parent is None or node.parent == self.NIL
    
    def is_left_child(self, node):
        """Check if node is a left child."""
        return node.parent and node == node.parent.left
    
    def is_right_child(self, node):
        """Check if node is a right child."""
        return node.parent and node == node.parent.right
    
    def get_uncle(self, node):
        """Get the uncle of a node (parent's sibling)."""
        if node.parent is None or node.parent.parent is None:
            return None
        grandparent = node.parent.parent
        if self.is_left_child(node.parent):
            return grandparent.right
        else:
            return grandparent.left
    
    def get_grandparent(self, node):
        """Get the grandparent of a node."""
        if node.parent:
            return node.parent.parent
        return None
    
    # ==================== Rotation Methods ====================
    
    def rotate_left(self, x):
        """
        Left rotation around node x.
        
              x                    y
             / \                  / \
            a   y      -->       x   c
               / \              / \
              b   c            a   b
        """
        y = x.right
        
        # Turn y's left subtree into x's right subtree
        x.right = y.left
        if not self.is_nil(y.left):
            y.left.parent = x
        
        # Link x's parent to y
        y.parent = x.parent
        if x.parent is None:
            self.root = y
        elif self.is_left_child(x):
            x.parent.left = y
        else:
            x.parent.right = y
        
        # Put x on y's left
        y.left = x
        x.parent = y
        
        return y
    
    def rotate_right(self, y):
        """
        Right rotation around node y.
        
              y                  x
             / \                / \
            x   c     -->      a   y
           / \                    / \
          a   b                  b   c
        """
        x = y.left
        
        # Turn x's right subtree into y's left subtree
        y.left = x.right
        if not self.is_nil(x.right):
            x.right.parent = y
        
        # Link y's parent to x
        x.parent = y.parent
        if y.parent is None:
            self.root = x
        elif self.is_right_child(y):
            y.parent.right = x
        else:
            y.parent.left = x
        
        # Put y on x's right
        x.right = y
        y.parent = x
        
        return x
    
    # ==================== Insert Methods ====================
    
    def insert(self, data):
        """Insert a new value into the Red-Black Tree."""
        # Create new RED node
        new_node = RBNode(data, color=RED)
        new_node.left = self.NIL
        new_node.right = self.NIL
        
        # Standard BST insert
        parent = None
        current = self.root
        
        while not self.is_nil(current):
            parent = current
            if data < current.data:
                current = current.left
            elif data > current.data:
                current = current.right
            else:
                return  # Duplicate, don't insert
        
        new_node.parent = parent
        
        if parent is None:
            self.root = new_node
        elif data < parent.data:
            parent.left = new_node
        else:
            parent.right = new_node
        
        # Fix Red-Black properties
        self._fix_insert(new_node)
    
    def _fix_insert(self, node):
        """
        Fix Red-Black Tree properties after insertion.
        Handles all three cases.
        """
        while node != self.root and node.parent.is_red():
            uncle = self.get_uncle(node)
            grandparent = self.get_grandparent(node)
            
            if self.is_left_child(node.parent):
                # Parent is left child of grandparent
                if uncle and uncle.is_red():
                    # Case 1: Uncle is RED → Recolor
                    node.parent.color = BLACK
                    uncle.color = BLACK
                    grandparent.color = RED
                    node = grandparent  # Move up to check grandparent
                else:
                    # Uncle is BLACK
                    if self.is_right_child(node):
                        # Case 2: Node is right child → Left rotate parent
                        node = node.parent
                        self.rotate_left(node)
                    # Case 3: Node is left child → Right rotate grandparent
                    node.parent.color = BLACK
                    grandparent.color = RED
                    self.rotate_right(grandparent)
            else:
                # Parent is right child of grandparent (mirror cases)
                if uncle and uncle.is_red():
                    # Case 1: Uncle is RED → Recolor
                    node.parent.color = BLACK
                    uncle.color = BLACK
                    grandparent.color = RED
                    node = grandparent
                else:
                    # Uncle is BLACK
                    if self.is_left_child(node):
                        # Case 2: Node is left child → Right rotate parent
                        node = node.parent
                        self.rotate_right(node)
                    # Case 3: Node is right child → Left rotate grandparent
                    node.parent.color = BLACK
                    grandparent.color = RED
                    self.rotate_left(grandparent)
        
        # Ensure root is always BLACK
        self.root.color = BLACK
    
    # ==================== Search Method ====================
    
    def search(self, data):
        """Search for a value in the tree."""
        current = self.root
        while not self.is_nil(current):
            if data == current.data:
                return current
            elif data < current.data:
                current = current.left
            else:
                current = current.right
        return None
    
    # ==================== Traversal Methods ====================
    
    def inorder(self):
        """Return inorder traversal as list."""
        result = []
        self._inorder_helper(self.root, result)
        return result
    
    def _inorder_helper(self, node, result):
        if not self.is_nil(node):
            self._inorder_helper(node.left, result)
            result.append(node)
            self._inorder_helper(node.right, result)
    
    # ==================== Verification Methods ====================
    
    def verify_properties(self):
        """Verify all Red-Black Tree properties."""
        if self.is_nil(self.root):
            return True, "Empty tree is valid"
        
        errors = []
        
        # Property 2: Root is BLACK
        if self.root.is_red():
            errors.append("Property 2 violated: Root is RED")
        
        # Property 4: No red-red & Property 5: Black-height
        black_heights = []
        self._check_properties(self.root, 0, black_heights, errors)
        
        # Check all black heights are equal
        if len(set(black_heights)) > 1:
            errors.append(f"Property 5 violated: Unequal black heights {black_heights}")
        
        if errors:
            return False, errors
        return True, f"Valid Red-Black Tree (black-height: {black_heights[0] if black_heights else 0})"
    
    def _check_properties(self, node, black_count, black_heights, errors):
        """Recursively check properties 4 and 5."""
        if self.is_nil(node):
            black_heights.append(black_count)
            return
        
        # Property 4: Check for red-red violation
        if node.is_red():
            if (not self.is_nil(node.left) and node.left.is_red()):
                errors.append(f"Property 4 violated: RED-RED at {node} → {node.left}")
            if (not self.is_nil(node.right) and node.right.is_red()):
                errors.append(f"Property 4 violated: RED-RED at {node} → {node.right}")
        
        # Count black nodes
        new_count = black_count + (1 if node.is_black() else 0)
        self._check_properties(node.left, new_count, black_heights, errors)
        self._check_properties(node.right, new_count, black_heights, errors)
    
    def black_height(self):
        """Calculate the black-height of the tree."""
        count = 0
        node = self.root
        while not self.is_nil(node):
            if node.is_black():
                count += 1
            node = node.left
        return count
    
    # ==================== Display Methods ====================
    
    def print_tree(self):
        """Print the tree structure with colors."""
        if self.is_nil(self.root):
            print("<Empty Tree>")
            return
        self._print_tree_helper(self.root, "", True)
    
    def _print_tree_helper(self, node, prefix, is_last):
        """Helper for printing tree structure."""
        if self.is_nil(node):
            return
        
        connector = "└── " if is_last else "├── "
        color = "R" if node.is_red() else "B"
        print(f"{prefix}{connector}({color}){node.data}")
        
        new_prefix = prefix + ("    " if is_last else "│   ")
        
        # Check if we have children to print
        has_left = not self.is_nil(node.left)
        has_right = not self.is_nil(node.right)
        
        if has_left or has_right:
            if has_right:
                self._print_tree_helper(node.right, new_prefix, False)
            if has_left:
                self._print_tree_helper(node.left, new_prefix, True)

---

## 5. Insert with Color Fixing

When we insert a new node (always RED initially), we may violate Property 4 (no red-red). There are **3 cases** to handle:

### Insert Cases Overview

```
New node is always inserted as RED
              ↓
    ┌─────────────────────┐
    │ Is parent BLACK?    │
    └─────────────────────┘
         │YES        │NO (RED-RED violation!)
         ↓           ↓
      DONE!    ┌─────────────────┐
               │ What color is   │
               │ uncle?          │
               └─────────────────┘
                 │RED        │BLACK
                 ↓           ↓
              CASE 1     ┌────────────────┐
             (recolor)   │ Inner or outer │
                         │ grandchild?    │
                         └────────────────┘
                           │INNER    │OUTER
                           ↓         ↓
                        CASE 2    CASE 3
                       (rotate   (rotate +
                        to case3) recolor)
```

---

### Case 1: Uncle is RED (Recolor)

```
Before:                      After:
        (B)G                       (R)G ← recolor, might need fix upward
       /    \                     /    \
     (R)P   (R)U        →      (B)P   (B)U
     /                         /
   (R)N                      (R)N

Action: Recolor P → BLACK, U → BLACK, G → RED
Then: Continue fixing at G (might have new violation)
```

---

### Case 2: Uncle is BLACK, Node is Inner Child (Rotate to Case 3)

```
Before (N is inner child):    After (now Case 3):
        (B)G                       (B)G
       /    \                     /    \
     (R)P   (B)U        →      (R)N   (B)U
        \                      /
       (R)N                  (R)P

Action: Left-rotate P (making N the new parent of P)
Then: This becomes Case 3 with P as the new node
```

---

### Case 3: Uncle is BLACK, Node is Outer Child (Rotate + Recolor)

```
Before (N is outer child):    After (DONE!):
          (B)G                      (B)P
         /    \                    /    \
       (R)P   (B)U      →       (R)N   (R)G
       /                                  \
     (R)N                                (B)U

Action: Right-rotate G, swap colors P↔G
Result: Tree is balanced!
```

---

## 6. Rotations with Color Changes

### Left Rotation

```
Left rotation around X:

        X                          Y
       / \                        / \
      a   Y          →           X   c
         / \                    / \
        b   c                  a   b

Steps:
1. Y becomes the new root of this subtree
2. X becomes Y's left child
3. Y's left child (b) becomes X's right child
```

### Right Rotation

```
Right rotation around Y:

          Y                      X
         / \                    / \
        X   c        →         a   Y
       / \                        / \
      a   b                      b   c

Steps:
1. X becomes the new root of this subtree
2. Y becomes X's right child
3. X's right child (b) becomes Y's left child
```

### Rotation with Color Changes (Case 3 Example)

```
Before:                    After:
      (B)20                     (B)10
      /                         /    \
   (R)10              →      (R)5   (R)20
   /
(R)5

Right-rotate 20, then:
- 10 becomes BLACK (was RED)
- 20 becomes RED (was BLACK)
```

---

## 7. Examples

Let's build a tree step by step and see each case being triggered.

In [ ]:
# Example 1: Simple insertion sequence
print("=" * 60)
print("Example 1: Insert 10, 20, 30, 15")
print("=" * 60)

tree = RedBlackTree()

print("\nStep 1: Insert 10")
print("-" * 30)
tree.insert(10)
tree.print_tree()
print(f"Verification: {tree.verify_properties()}")
print("Note: First node becomes BLACK root")

In [ ]:
print("\nStep 2: Insert 20")
print("-" * 30)
tree.insert(20)
tree.print_tree()
print(f"Verification: {tree.verify_properties()}")
print("Note: 20 is RED, parent 10 is BLACK → No fix needed")

In [ ]:
print("\nStep 3: Insert 30 (triggers Case 3!)")
print("-" * 30)
print("Before fix:")
print("""    (B)10
        \\
        (R)20
            \\
           (R)30  ← RED-RED violation!""")
print("\nUncle (10's left) is NIL (BLACK)")
print("30 is outer grandchild → Case 3: Rotate + Recolor")
print("\nAfter fix:")
tree.insert(30)
tree.print_tree()
print(f"Verification: {tree.verify_properties()}")

In [ ]:
print("\nStep 4: Insert 15 (triggers Case 1!)")
print("-" * 30)
print("Before fix:")
print("""        (B)20
       /    \\
    (R)10  (R)30
        \\
       (R)15  ← RED-RED violation!""")
print("\nUncle (30) is RED → Case 1: Recolor")
print("  - 10, 30 → BLACK")
print("  - 20 → RED (but it's root, so stays BLACK)")
print("\nAfter fix:")
tree.insert(15)
tree.print_tree()
print(f"Verification: {tree.verify_properties()}")

In [ ]:
# Example 2: Demonstrating Case 2 (inner grandchild)
print("\n" + "=" * 60)
print("Example 2: Demonstrating Case 2 (inner grandchild)")
print("=" * 60)

tree2 = RedBlackTree()
tree2.insert(30)
tree2.insert(20)
print("\nAfter inserting 30, 20:")
tree2.print_tree()

print("\nInsert 25 (inner grandchild - Case 2 then Case 3):")
print("Before:")
print("""    (B)30
    /
 (R)20
     \\
    (R)25  ← Inner grandchild, Uncle is NIL (BLACK)""")
print("\nCase 2: Left-rotate 20 to make it outer grandchild")
print("Then Case 3: Right-rotate 30 + recolor")
print("\nAfter:")
tree2.insert(25)
tree2.print_tree()
print(f"Verification: {tree2.verify_properties()}")

In [ ]:
# Example 3: Building a larger tree
print("\n" + "=" * 60)
print("Example 3: Building a larger tree")
print("=" * 60)

tree3 = RedBlackTree()
values = [13, 8, 17, 1, 11, 15, 25, 6, 22, 27]
print(f"\nInserting values: {values}")

for val in values:
    tree3.insert(val)

print("\nFinal Tree:")
tree3.print_tree()
print(f"\nVerification: {tree3.verify_properties()}")
print(f"Black-height: {tree3.black_height()}")
print(f"Inorder traversal: {[str(n) for n in tree3.inorder()]}")

In [ ]:
# Example 4: Verify black-height property
print("\n" + "=" * 60)
print("Example 4: Verifying black-height property")
print("=" * 60)

def print_all_paths_with_black_count(tree, node, path, black_count):
    """Print all root-to-leaf paths with black node count."""
    if tree.is_nil(node):
        path_str = " → ".join(path)
        print(f"  {path_str} → NIL : black_count = {black_count}")
        return
    
    color = "B" if node.is_black() else "R"
    new_path = path + [f"({color}){node.data}"]
    new_count = black_count + (1 if node.is_black() else 0)
    
    print_all_paths_with_black_count(tree, node.left, new_path, new_count)
    print_all_paths_with_black_count(tree, node.right, new_path, new_count)

print("\nAll root-to-leaf paths in tree3:")
print_all_paths_with_black_count(tree3, tree3.root, [], 0)
print("\n✓ All paths have the same black count!")

In [ ]:
# Example 5: Search demonstration
print("\n" + "=" * 60)
print("Example 5: Search operations")
print("=" * 60)

search_values = [17, 6, 100, 13]
for val in search_values:
    result = tree3.search(val)
    if result:
        print(f"  Search({val}): Found {result}")
    else:
        print(f"  Search({val}): Not found")

### Full Insertion Example (Step by Step)

```
Insert: 10, 20, 30, 15

Step 1: Insert 10       Step 2: Insert 20
    (B)10                  (B)10
                               \
                              (R)20

Step 3: Insert 30 - triggers rebalancing!
    (B)10
        \
        (R)20
            \
           (R)30  ← RED-RED violation!

Fix: Left rotate 10, recolor
        (B)20
       /    \
    (R)10  (R)30

Step 4: Insert 15
        (B)20
       /    \
    (R)10  (R)30
        \
       (R)15  ← RED-RED violation!

Fix: Recolor (uncle is RED)
        (R)20  ← but root must be black!
       /    \
    (B)10  (B)30
        \
       (R)15

Final fix: Make root black
        (B)20
       /    \
    (B)10  (B)30
        \
       (R)15
```

---

## 8. Complexity Analysis

### Time Complexity

| Operation | Time Complexity | Notes |
|-----------|-----------------|-------|
| **Search** | O(log n) | Same as BST, guaranteed by balance |
| **Insert** | O(log n) | BST insert + at most O(log n) fix-ups |
| **Delete** | O(log n) | BST delete + at most O(log n) fix-ups |
| **Rotation** | O(1) | Constant pointer updates |
| **Recoloring** | O(1) | Constant color changes |

### Space Complexity

| Aspect | Space | Notes |
|--------|-------|-------|
| **Tree storage** | O(n) | One node per element |
| **Per node** | O(1) | data + color + 3 pointers |
| **Recursive operations** | O(log n) | Stack depth for traversals |

### Why O(log n) is Guaranteed

```
Key insight: Black-height property ensures balance!

If black-height = bh, then:
  - Minimum nodes: 2^bh - 1 (all black path)
  - Maximum height: 2 * bh (alternating red-black)

For n nodes:
  n ≥ 2^bh - 1
  bh ≤ log₂(n+1)
  height ≤ 2 * bh ≤ 2 * log₂(n+1)

Therefore: height = O(log n)
```

### Comparison Summary

```
                AVL Tree          Red-Black Tree
              ┌─────────────┐    ┌─────────────┐
  Search      │  O(log n)   │    │  O(log n)   │ ← RB slightly slower
  Insert      │  O(log n)   │    │  O(log n)   │ ← RB fewer rotations
  Delete      │  O(log n)   │    │  O(log n)   │ ← RB fewer rotations
  Height      │ ≤1.44 log n │    │  ≤2 log n   │ ← AVL shorter
              └─────────────┘    └─────────────┘
```

---

## Summary

### Key Takeaways

1. **5 Properties** ensure the tree stays balanced:
   - Nodes are RED or BLACK
   - Root is BLACK
   - NIL leaves are BLACK
   - No red-red parent-child
   - Equal black-height on all paths

2. **Insert Fix-up** has 3 cases:
   - Case 1: Uncle RED → Recolor
   - Case 2: Uncle BLACK, inner child → Rotate to Case 3
   - Case 3: Uncle BLACK, outer child → Rotate + Recolor

3. **Trade-offs vs AVL**:
   - Red-Black: Faster inserts/deletes, used in language libraries
   - AVL: Faster lookups, stricter balance

4. **Real-world uses**:
   - Java: `TreeMap`, `TreeSet`
   - C++: `std::map`, `std::set`
   - Linux kernel: Completely Fair Scheduler

---

[← Previous: AVL Trees](../../Tier_4_Algorithms_and_Data_Structures/05_Tree_Structures/02_avl_trees.ipynb) | [Next: Hash Tables and Bloom Filters →](../../Tier_4_Algorithms_and_Data_Structures/06_Hash_Based_Structures/01_hash_tables_bloom.ipynb)